# Demo Notebook: Document Forgery & Tampering Detection Pipeline

This notebook demonstrates the stable `ml-first-api` branch of the document forgery detection project.

It covers:

- loading the generated feature store,
- checking dataset distribution,
- inspecting extracted forensic features,
- running the rule-based evidence generator,
- running the trained ML model on one feature row,
- showing how to start and test the FastAPI endpoint.

Note: The dataset used in this project is synthetic. In this context, `Authentic` means an untampered synthetic/base document, and `Forged` means a deliberately modified synthetic document. The system detects tampering patterns; it does not legally verify real-world documents.


## 1. Project Structure

Expected relevant files:

```txt
app/
├── analyzer.py
├── evidence_generator.py
├── feature_extractor.py
├── main.py
└── model.py

data/
├── labels.csv
└── feature_store.csv

train_model.py
requirements.txt
README.md
reports/test_report.md
```


## 2. Import Required Libraries


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Make sure the notebook can import from the project root.
# If running from notebooks/, move one level up.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.append(str(project_root))
os.chdir(project_root)

print("Project root:", project_root)


## 3. Load Feature Store

The feature store is generated by:

```bash
python -m app.analyzer
```

It should be available at:

```txt
data/feature_store.csv
```


In [ ]:
feature_store_path = Path("data/feature_store.csv")

if not feature_store_path.exists():
    raise FileNotFoundError(
        "data/feature_store.csv not found. Run: python -m app.analyzer"
    )

df = pd.read_csv(feature_store_path)
print("Feature store shape:", df.shape)
df.head()


## 4. Dataset Distribution


In [ ]:
print("Label distribution:")
print(df["label"].value_counts())

print("\nCategory distribution:")
print(df["category"].value_counts())


## 5. Inspect Feature Columns

The stable model uses numeric forensic features such as brightness, noise, sharpness, compression, contour, metadata, and structural PDF features.


In [ ]:
non_feature_cols = ["file_path", "file_name", "file_extension", "label", "category"]

feature_df = df.drop(columns=non_feature_cols, errors="ignore")
numeric_features = feature_df.select_dtypes(include=["int64", "float64", "int32", "float32"])

print("Number of numeric features:", len(numeric_features.columns))
print("\nFeature columns:")
for col in numeric_features.columns:
    print("-", col)


## 6. Run Rule-Based Evidence Generator

The evidence generator returns a rule-based authenticity score and human-readable anomaly explanations.


In [ ]:
from app.evidence_generator import generate_evidence

sample_row = df.iloc[0].to_dict()
evidence_result = generate_evidence(sample_row)
evidence_result


## 7. Run ML Prediction on a Feature Row

The trained model is loaded from:

```txt
models/rf_model.pkl
models/feature_columns.pkl
```

If these files are missing, regenerate them by running:

```bash
python train_model.py
```


In [ ]:
from app.model import predict_with_model

try:
    prediction_result = predict_with_model(sample_row)
    prediction_result
except FileNotFoundError as e:
    print("Model files not found. Run: python train_model.py")
    print(e)


## 8. Train the Baseline Model

To train the model from the terminal:

```bash
python train_model.py
```

Expected stable-branch result from the recorded run:

```txt
Dataset shape: (70, 28)
Label distribution:
Forged       36
Authentic    34

Number of training features: 23
Test Accuracy: 0.7777777777777778
Mean CV Accuracy: 0.9
```


## 9. Run FastAPI Microservice

Start the API from the project root:

```bash
uvicorn app.main:app --reload
```

Open Swagger UI:

```txt
http://127.0.0.1:8000/docs
```

Use:

```txt
POST /doc/analyze
```

Upload a `.jpg`, `.jpeg`, `.png`, or `.pdf` document.


## 10. Sample API Response

Recorded forged PAN-card-style sample from the stable branch:

```json
{
  "filename": "AIGen_PAN Card_4.png",
  "authenticity_score": 1,
  "tampering_score": 99,
  "risk_label": "Forged",
  "final_authenticity_score": 1,
  "final_tampering_risk_score": 99,
  "ml_prediction": "Forged",
  "ml_confidence": 0.9868,
  "class_probabilities": {
    "Authentic": 0.013179141564516904,
    "Forged": 0.986820858435483
  },
  "rule_based_authenticity_score": 85,
  "rule_based_tampering_risk_score": 15,
  "rule_based_label": "Authentic",
  "processing_time_seconds": 3.5328
}
```


## 11. Interpretation

The stable pipeline uses an ML-first decision strategy:

- the RandomForest classifier provides the primary `Authentic`/`Forged` prediction,
- rule-based evidence provides supporting forensic explanation,
- the API returns both ML outputs and forensic evidence so the decision is interpretable.

In the recorded forged PAN example, the ML model predicted `Forged` with high confidence, while the rule-based module detected a compression anomaly. The final API decision was therefore `Forged`.


## 12. Limitations and Future Work

Current limitations:

- dataset is small and synthetic,
- model does not verify legal genuineness,
- OCR-based semantic validation is not part of the stable `ml-first-api` branch,
- advanced suspicious-region heatmaps are not implemented,
- digital signature validation is not cryptographically implemented,
- font inconsistency detection is limited in image-only documents.

Future work:

- OCR-based Aadhaar/PAN validation,
- condition-wise tampering labels,
- larger and more diverse dataset,
- better local suspicious-region detection,
- separate specialized models per document type,
- stronger metadata and PDF-layer forensics.
